In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pathlib import Path
import os
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="pyspark")

current_path = os.getcwd()
project_root = Path(current_path).parent
os.chdir(project_root)
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
spark = SparkSession.builder \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

cnefe_df_saude = spark.read.parquet("data/integrated/integrated_cnefe_addresses.parquet") \
                    .filter(F.col("COD_ESPECIE") == 5)
                    
cnes_df = spark.read.csv("data/raw/cnes/BASE_DE_DADOS_CNES_202606/tbEstabelecimento202606.csv", header=True, inferSchema=True, sep=";")

print("CNEFE  rows:", cnefe_df_saude.count())
print("CNES  rows:", cnes_df.count())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/31 15:20:27 WARN Utils: Your hostname, lucas-vital-Q570M-D3H, resolves to a loopback address: 127.0.1.1; using 192.168.100.25 instead (on interface wlp8s0)
26/07/31 15:20:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/lucas-vital/projects/brazilian_address_linkage/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/07/31 15:20:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


CNEFE  rows: 247510
CNES  rows: 627705


In [2]:
cnefe_df_saude.show(5)

26/07/31 15:20:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------------+------+-------------+------------+---------------+----------------+----------+--------+--------+--------------+----------------+------------------+--------------------+------------+---------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+----------+----------+------------+-----------+--------------------+----------------------------+----------------------------+------------------------------+---------------+---+
|COD_UNICO_ENDERECO|COD_UF|COD_MUNICIPIO|COD_DISTRITO|COD_SUBDISTRITO|       COD_SETOR|NUM_QUADRA|NUM_FACE|     CEP|DSC_LOCALIDADE|NOM_TIPO_SEGLOGR|NOM_TITULO_SEGLOGR|         NOM_SEGLOGR|NUM_ENDERECO|DSC_MODIFICADOR|NOM_COMP_ELEM1|VAL_COMP_ELEM1|NOM_COMP_ELEM2|VAL_COMP_ELEM2|NOM_COMP_ELEM3|VAL_COMP_ELEM3|NOM_COMP_ELEM4|VAL_COMP_ELEM4|NOM_COMP_ELEM5|VAL_COMP_ELEM5|  LATITUDE| LONGITUDE|NV_GEO_COORD|COD_ESPECIE| DSC_ESTABELECIMENTO|COD_INDICADOR_ESTAB_

In [3]:
cnes_df.show(5)

+-------------+-------+-------------------+-------+---------+--------------------+--------------------+--------------------+-----------+--------------+----------------+--------+---------------+---------------+---------------------+--------------------------+--------------+------+--------------------+------+--------------+------------+------------+-------------------+--------------------+------------------+---------------+-----------+----------+--------------------+----------------+-------------------+------------------------------------+----------+----------------+--------------+-------------------+---------------+------+-----------+------------+--------------------------------+--------------+---------------+----------------------+---------------------------+-------------------+---------------+-----------------+---------+-------------------------------------------+-----------------------+----------------------+-----------------------+-------------------+------------+
|   CO_UNIDADE|CO_

In [15]:
cnes_df.agg(
    F.countDistinct("CO_UNIDADE")).show()

+--------------------------+
|count(DISTINCT CO_UNIDADE)|
+--------------------------+
|                    627705|
+--------------------------+



In [4]:
cnefe_df_saude.printSchema()

root
 |-- COD_UNICO_ENDERECO: long (nullable = true)
 |-- COD_UF: integer (nullable = true)
 |-- COD_MUNICIPIO: long (nullable = true)
 |-- COD_DISTRITO: long (nullable = true)
 |-- COD_SUBDISTRITO: long (nullable = true)
 |-- COD_SETOR: string (nullable = true)
 |-- NUM_QUADRA: integer (nullable = true)
 |-- NUM_FACE: integer (nullable = true)
 |-- CEP: string (nullable = true)
 |-- DSC_LOCALIDADE: string (nullable = true)
 |-- NOM_TIPO_SEGLOGR: string (nullable = true)
 |-- NOM_TITULO_SEGLOGR: string (nullable = true)
 |-- NOM_SEGLOGR: string (nullable = true)
 |-- NUM_ENDERECO: integer (nullable = true)
 |-- DSC_MODIFICADOR: string (nullable = true)
 |-- NOM_COMP_ELEM1: string (nullable = true)
 |-- VAL_COMP_ELEM1: string (nullable = true)
 |-- NOM_COMP_ELEM2: string (nullable = true)
 |-- VAL_COMP_ELEM2: string (nullable = true)
 |-- NOM_COMP_ELEM3: string (nullable = true)
 |-- VAL_COMP_ELEM3: string (nullable = true)
 |-- NOM_COMP_ELEM4: string (nullable = true)
 |-- VAL_COMP_ELE

In [5]:
cnefe_df_saude.select("DSC_ESTABELECIMENTO", "DSC_MODIFICADOR").distinct().toPandas().to_csv("data/outputs/cnefe_estabelecimentos.csv", index=False)

/home/lucas-vital/projects/brazilian_address_linkage/.venv/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [6]:
cnes_df.printSchema()

root
 |-- CO_UNIDADE: string (nullable = true)
 |-- CO_CNES: integer (nullable = true)
 |-- NU_CNPJ_MANTENEDORA: long (nullable = true)
 |-- TP_PFPJ: integer (nullable = true)
 |-- NIVEL_DEP: integer (nullable = true)
 |-- NO_RAZAO_SOCIAL: string (nullable = true)
 |-- NO_FANTASIA: string (nullable = true)
 |-- NO_LOGRADOURO: string (nullable = true)
 |-- NU_ENDERECO: string (nullable = true)
 |-- NO_COMPLEMENTO: string (nullable = true)
 |-- NO_BAIRRO: string (nullable = true)
 |-- CO_CEP: integer (nullable = true)
 |-- CO_REGIAO_SAUDE: string (nullable = true)
 |-- CO_MICRO_REGIAO: string (nullable = true)
 |-- CO_DISTRITO_SANITARIO: string (nullable = true)
 |-- CO_DISTRITO_ADMINISTRATIVO: string (nullable = true)
 |-- NU_TELEFONE: string (nullable = true)
 |-- NU_FAX: string (nullable = true)
 |-- NO_EMAIL: string (nullable = true)
 |-- NU_CPF: long (nullable = true)
 |-- NU_CNPJ: long (nullable = true)
 |-- CO_ATIVIDADE: integer (nullable = true)
 |-- CO_CLIENTELA: integer (nullab

## Preenchimento de campos

In [26]:
from pyspark.sql import functions as F

total_rows = cnes_df.count()

fulfillment_exprs = [
    F.struct(
        F.lit(c).alias("column"),
        (F.count(F.col(c)) / F.lit(total_rows) * 100).alias("fulfillment_pct")
    )
    for c in cnes_df.columns
]

result = cnes_df.select(F.array(*fulfillment_exprs).alias("stats")) \
    .selectExpr("explode(stats) as stats") \
    .select("stats.column", "stats.fulfillment_pct")

result.orderBy(F.col("fulfillment_pct").desc()).show(cnes_df.columns.__len__(), truncate=False)

+-------------------------------------------+---------------------+
|column                                     |fulfillment_pct      |
+-------------------------------------------+---------------------+
|CO_UNIDADE                                 |100.0                |
|CO_CNES                                    |100.0                |
|TP_PFPJ                                    |100.0                |
|NIVEL_DEP                                  |100.0                |
|NO_LOGRADOURO                              |100.0                |
|CO_CEP                                     |100.0                |
|CO_ATIVIDADE                               |100.0                |
|TP_UNIDADE                                 |100.0                |
|CO_ESTADO_GESTOR                           |100.0                |
|CO_MUNICIPIO_GESTOR                        |100.0                |
|TO_CHAR(DT_ATUALIZACAO,'DD/MM/YYYY')       |100.0                |
|CO_USUARIO                                 |100

## Tentativas de pareamento

### Por latitude e longitude

In [7]:
from pyspark.sql import functions as F

cnes_df.filter(F.col("CO_CEP") == "29060120").show(11, truncate=False)

+-------------+-------+-------------------+-------+---------+----------------------------------------------------------+------------------------------------------------+------------------------------------+-----------+-------------------+----------------+--------+---------------+---------------+---------------------+--------------------------+--------------------+------+--------------------------------------+-----------+--------------+------------+------------+------------+--------------------+------------------+--------------------+-----------+----------+--------------------+----------------+-------------------+------------------------------------+-----------+----------------+--------------+-------------------+---------------+------+-------------------+-------------------+--------------------------------+--------------+---------------+----------------------+---------------------------+-------------------+---------------+-----------------+---------+-------------------------------------

In [8]:
# Run this fresh, after restarting the kernel and re-loading cnes_df / cnefe_df_saude from source

from pyspark.sql import functions as F

NUMERIC_PATTERN = r'^-?\d+(\.\d+)?([eE][+-]?\d+)?$'

def safe_double(col_name):
    col = F.col(col_name)
    col = F.trim(col)
    # Replace comma decimal separator with dot (common in Brazilian data)
    col = F.regexp_replace(col, ',', '.')
    # If the value ends in a bare dot (e.g. "12." from "12,"), append a 0 -> "12.0"
    col = F.regexp_replace(col, r'\.$', '.0')
    return F.when(col.rlike(NUMERIC_PATTERN), col.cast("double")).otherwise(F.lit(None).cast("double"))

cnes_df = cnes_df.withColumn("lat_d", safe_double("NU_LATITUDE")) \
                  .withColumn("lon_d", safe_double("NU_LONGITUDE"))

cnefe_df_saude = cnefe_df_saude.withColumn("lat_d", safe_double("LATITUDE")) \
                                .withColumn("lon_d", safe_double("LONGITUDE"))

cnes_df = cnes_df.cache()
cnefe_df_saude = cnefe_df_saude.cache()
cnes_df.count()
cnefe_df_saude.count()

bad_cnes = cnes_df.filter(F.col("lat_d").isNull() | F.col("lon_d").isNull())
n_bad = bad_cnes.count()
print("Registros CNES com lat/long malformados: %d" % n_bad)
if n_bad > 0:
    bad_cnes.select("NU_LATITUDE", "NU_LONGITUDE").distinct().show(30, truncate=False)

cnes_clean = cnes_df.filter(F.col("lat_d").isNotNull() & F.col("lon_d").isNotNull())
cnefe_clean = cnefe_df_saude.filter(F.col("lat_d").isNotNull() & F.col("lon_d").isNotNull())

joint_lat_long_df = cnes_clean.join(
    cnefe_clean,
    (cnes_clean.lat_d == cnefe_clean.lat_d) & (cnes_clean.lon_d == cnefe_clean.lon_d),
    "inner"
).cache()

matched = joint_lat_long_df.count()
total = cnes_df.count()

print("Número de registros encontrados no CNES: %d" % matched)
if total > 0:
    print("Cobertura: %0.6f %%" % (matched / total * 100))
else:
    print("Cobertura: N/A (cnes_df está vazio)")

Registros CNES com lat/long malformados: 58724
+-----------+------------+
|NU_LATITUDE|NU_LONGITUDE|
+-----------+------------+
|-38.532703 |03/09/2019  |
|NULL       |NULL        |
|NULL       |undefined   |
|NULL       |-47.6385644 |
|NULL       |-52.099     |
|NULL       |431229W     |
|NULL       |-34.8995092 |
|NULL       |-           |
+-----------+------------+



Número de registros encontrados no CNES: 1
Cobertura: 0.000159 %


### Por nome do estabelecimento

In [9]:
joint_nome_df = cnes_df.join(cnefe_df_saude, [cnes_df.NO_FANTASIA == cnefe_df_saude.DSC_ESTABELECIMENTO ], "inner")
print("Número de registros encontrados no CNES: %d" % joint_nome_df.count())
print("Número de registros no CNEFE: %d" % cnefe_df_saude.count())
print("Cobertura: %0.6f %%" % (joint_nome_df.count() / cnes_df.count() * 100))

Número de registros encontrados no CNES: 11144419
Número de registros no CNEFE: 247510
Cobertura: 1775.423009 %


Existem alguns nomes fantasias que se repetem mais de uma vez, alguns bem frequentes. Isso ocorre até mesmo em unidades de saúde em localidades próximas

In [25]:
cnes_df.groupBy(["NO_FANTASIA", "CO_CEP"]).count().orderBy(F.desc("count")).show(100, truncate=False)

+--------------------------------------------------------+--------+-----+
|NO_FANTASIA                                             |CO_CEP  |count|
+--------------------------------------------------------+--------+-----+
|CONSULTORIO ODONTOLOGICO                                |76400000|13   |
|LIDERA SAUDE                                            |57022180|11   |
|CONSULTORIO MEDICO                                      |76400000|10   |
|SAO JOAO FARMACIAS                                      |95555000|10   |
|CONSULTORIO MEDICO                                      |17250000|9    |
|SAMED                                                   |35960000|9    |
|FARMACIA DO POVO                                        |45810000|9    |
|CONSULTORIO ODONTOLOGICO                                |88220000|9    |
|CONSULTORIO ODONTOLOGICO                                |76970000|8    |
|SAO JOAO FARMACIAS                                      |98700000|8    |
|CLINICA BERGGASSE                    

### Por nome do estabelecimento e lougradouro

Aqui acredtio que consiga a cobertura a nível mais real

In [10]:
cnefe_df_saude = cnefe_df_saude.withColumn(
    "NO_LOGRADOURO",
    F.concat_ws(" ", F.col("NOM_TIPO_SEGLOGR"), F.col("NOM_TITULO_SEGLOGR"), F.col("NOM_SEGLOGR"))
)
joint_log_df = cnes_df.join(cnefe_df_saude, [cnes_df.NO_FANTASIA == cnefe_df_saude.DSC_ESTABELECIMENTO, cnes_df.NO_LOGRADOURO == cnefe_df_saude.NO_LOGRADOURO ], "inner")
print("Número de registros encontrados no CNES: %d" % joint_log_df.count())
print("Cobertura: %0.6f %%" % (joint_log_df.count() / cnes_df.count() * 100))

Número de registros encontrados no CNES: 7980
Cobertura: 1.271298 %


### Por nome do estabelecimento e lougradouro e numero

Aqui acredito demonstrar que existam erros no registro do numero em pelo menos uma das bases

In [11]:
joint_log_num_df = cnes_df.join(
    cnefe_df_saude,
    [
        cnes_df.NO_FANTASIA == cnefe_df_saude.DSC_ESTABELECIMENTO,
        cnes_df.NO_LOGRADOURO == cnefe_df_saude.NO_LOGRADOURO,
        F.col("NU_ENDERECO").cast("string") == F.col("NUM_ENDERECO").cast("string"),
    ],
    "inner"
)

print("Número de registros encontrados no CNES: %d" % joint_log_num_df.count())
print("Cobertura: %0.6f%%" % (joint_log_num_df.count() / cnes_df.count() * 100))

Número de registros encontrados no CNES: 3414
Cobertura: 0.543886%


In [12]:
joint_log_num_df = cnes_df.join(
    cnefe_df_saude,
    [
        cnes_df.NO_FANTASIA == cnefe_df_saude.DSC_ESTABELECIMENTO,
        cnes_df.CO_CEP == cnefe_df_saude.CEP,
    ],
    "inner"
)

print("Número de registros encontrados no CNES: %d" % joint_log_num_df.count())
print("Cobertura: %0.6f%%" % (joint_log_num_df.count() / cnes_df.count() * 100))

Número de registros encontrados no CNES: 11608
Cobertura: 1.849276%
